# Analisis Exploratorio: Dataset de Tarifas de Uber

En este notebook exploraremos el dataset de tarifas de Uber de Kaggle.

**Objetivo:** Entender la estructura de los datos, identificar patrones y anomalias, y preparar el terreno para la ingenieria de features.

**Dataset:** [Uber Fares Dataset](https://www.kaggle.com/datasets/yasserh/uber-fares-dataset)

## Contenido
1. Carga de datos
2. Analisis descriptivo
3. Deteccion de anomalias
4. Visualizaciones
5. Analisis temporal
6. Analisis geografico

In [ ]:
# Importaciones necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Carga de datos

Descarga el dataset de Kaggle y colocalo en `data/uber.csv` antes de ejecutar esta celda.

El dataset contiene viajes de Uber con informacion de coordenadas, fecha/hora, pasajeros y tarifa.

In [ ]:
# Cargar el dataset
df = pd.read_csv("../data/uber.csv")
print(f"Dimensiones del dataset: {df.shape}")
df.head()

## 2. Analisis descriptivo

Revisemos la estructura general: tipos de datos, valores faltantes y estadisticas basicas.

In [ ]:
# Informacion general del dataset
df.info()

In [ ]:
# Estadisticas descriptivas
df.describe().round(2)

In [ ]:
# Valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")
print(f"Porcentaje de filas con al menos un nulo: {df.isnull().any(axis=1).mean()*100:.2f}%")

## 3. Deteccion de anomalias

Busquemos valores que no tienen sentido: tarifas negativas, coordenadas imposibles, etc.

In [ ]:
# Tarifas anomalas
print("Tarifas anomalas:")
print(f"  Tarifas negativas: {(df['fare_amount'] < 0).sum()}")
print(f"  Tarifas = 0: {(df['fare_amount'] == 0).sum()}")
print(f"  Tarifas > $200: {(df['fare_amount'] > 200).sum()}")
print(f"  Tarifas > $500: {(df['fare_amount'] > 500).sum()}")

print(f"\nPasajeros anomalos:")
print(f"  Pasajeros = 0: {(df['passenger_count'] == 0).sum()}")
print(f"  Pasajeros > 6: {(df['passenger_count'] > 6).sum()}")

# Coordenadas fuera de rango
print(f"\nCoordenadas fuera de rango:")
print(f"  Latitud pickup fuera de [-90,90]: {(~df['pickup_latitude'].between(-90,90)).sum()}")
print(f"  Longitud pickup fuera de [-180,180]: {(~df['pickup_longitude'].between(-180,180)).sum()}")

## 4. Visualizaciones

### 4.1 Distribucion de la tarifa (variable objetivo)

In [ ]:
# Distribucion de tarifas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sin filtrar
axes[0].hist(df["fare_amount"], bins=100, color="#1f77b4", edgecolor="k", alpha=0.7)
axes[0].set_title("Todas las tarifas")
axes[0].set_xlabel("Tarifa (USD)")

# Filtradas $0-$100
df_clean = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 100)]
axes[1].hist(df_clean["fare_amount"], bins=80, color="#2ca02c", edgecolor="k", alpha=0.7)
axes[1].set_title("Tarifas $0 - $100")
axes[1].set_xlabel("Tarifa (USD)")

# Boxplot
axes[2].boxplot(df_clean["fare_amount"], vert=True)
axes[2].set_title("Boxplot de tarifas (filtradas)")
axes[2].set_ylabel("Tarifa (USD)")

plt.tight_layout()
plt.show()

### 4.2 Distribucion de pasajeros

In [ ]:
# Pasajeros vs tarifa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean["passenger_count"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color="#ff7f0e", edgecolor="k"
)
axes[0].set_title("Distribucion de Pasajeros")
axes[0].set_xlabel("Numero de Pasajeros")

sns.boxplot(
    data=df_clean[df_clean["passenger_count"].between(1, 6)],
    x="passenger_count", y="fare_amount", ax=axes[1], palette="Set2"
)
axes[1].set_title("Tarifa por Numero de Pasajeros")
axes[1].set_xlabel("Pasajeros")
axes[1].set_ylabel("Tarifa (USD)")

plt.tight_layout()
plt.show()

print("Observacion: El numero de pasajeros no parece afectar significativamente la tarifa.")

## 5. Analisis temporal

In [ ]:
# Convertir pickup_datetime
df_clean = df_clean.copy()
df_clean["pickup_datetime"] = pd.to_datetime(df_clean["pickup_datetime"], errors="coerce")
df_clean["hora"] = df_clean["pickup_datetime"].dt.hour
df_clean["dia_semana"] = df_clean["pickup_datetime"].dt.day_name()
df_clean["mes"] = df_clean["pickup_datetime"].dt.month

In [ ]:
# Tarifa por hora del dia y por dia de la semana
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Por hora
tarifa_hora = df_clean.groupby("hora")["fare_amount"].agg(["mean", "count"])
axes[0].bar(tarifa_hora.index, tarifa_hora["mean"], color="#9467bd", edgecolor="k")
axes[0].set_title("Tarifa Promedio por Hora")
axes[0].set_xlabel("Hora del dia")
axes[0].set_ylabel("Tarifa promedio (USD)")

# Por dia de la semana
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
tarifa_dia = df_clean.groupby("dia_semana")["fare_amount"].mean().reindex(orden_dias)
axes[1].bar(range(7), tarifa_dia.values, color="#d62728", edgecolor="k")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(["Lun", "Mar", "Mie", "Jue", "Vie", "Sab", "Dom"])
axes[1].set_title("Tarifa Promedio por Dia de la Semana")
axes[1].set_ylabel("Tarifa promedio (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Volumen de viajes por hora
fig, ax = plt.subplots(figsize=(12, 5))
viajes_hora = df_clean.groupby("hora").size()
ax.bar(viajes_hora.index, viajes_hora.values, color="#17becf", edgecolor="k")
ax.set_title("Volumen de Viajes por Hora del Dia")
ax.set_xlabel("Hora")
ax.set_ylabel("Numero de viajes")

# Marcar horas pico
for h in [7, 8, 9, 17, 18, 19]:
    ax.axvspan(h - 0.4, h + 0.4, alpha=0.2, color="red")
ax.legend(["Hora pico"], loc="upper left")

plt.tight_layout()
plt.show()

print("Las zonas rojas indican las horas pico tipicas (7-9 AM, 5-7 PM).")

## 6. Analisis geografico

In [ ]:
# Calcular distancia para el analisis
import sys
sys.path.insert(0, "..")
from src.data import haversine_distance

df_geo = df_clean.dropna(subset=["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]).copy()
df_geo = df_geo[
    (df_geo["pickup_latitude"].between(40.5, 41.0)) &
    (df_geo["pickup_longitude"].between(-74.3, -73.7))
]

df_geo["distancia_km"] = haversine_distance(
    df_geo["pickup_latitude"], df_geo["pickup_longitude"],
    df_geo["dropoff_latitude"], df_geo["dropoff_longitude"]
)

# Filtrar distancias razonables
df_geo = df_geo[(df_geo["distancia_km"] > 0) & (df_geo["distancia_km"] < 100)]

In [ ]:
# Relacion distancia vs tarifa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df_geo.sample(min(5000, len(df_geo)), random_state=42)

axes[0].scatter(sample["distancia_km"], sample["fare_amount"], alpha=0.3, s=10)
axes[0].set_title("Tarifa vs Distancia")
axes[0].set_xlabel("Distancia (km)")
axes[0].set_ylabel("Tarifa (USD)")
axes[0].set_xlim(0, 50)
axes[0].set_ylim(0, 100)

# Tarifa por km
df_geo["tarifa_por_km"] = df_geo["fare_amount"] / df_geo["distancia_km"]
tarifa_km_razonable = df_geo[df_geo["tarifa_por_km"].between(0, 20)]["tarifa_por_km"]
axes[1].hist(tarifa_km_razonable, bins=50, color="#e377c2", edgecolor="k", alpha=0.7)
axes[1].set_title("Distribucion de Tarifa por Km")
axes[1].set_xlabel("USD/km")
axes[1].set_ylabel("Frecuencia")
axes[1].axvline(tarifa_km_razonable.median(), color="red", linestyle="--", label=f"Mediana: ${tarifa_km_razonable.median():.2f}/km")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Distancia promedio: {df_geo['distancia_km'].mean():.2f} km")
print(f"Tarifa promedio por km: ${tarifa_km_razonable.median():.2f}")

In [ ]:
# Mapa de calor de puntos de recogida (zona de NYC)
fig, ax = plt.subplots(figsize=(10, 10))
sample_geo = df_geo.sample(min(10000, len(df_geo)), random_state=42)

ax.scatter(
    sample_geo["pickup_longitude"], sample_geo["pickup_latitude"],
    c=sample_geo["fare_amount"], cmap="YlOrRd", alpha=0.3, s=5, vmin=0, vmax=50
)
ax.set_title("Puntos de Recogida (color = tarifa)")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_xlim(-74.05, -73.75)
ax.set_ylim(40.63, 40.85)
plt.colorbar(ax.collections[0], label="Tarifa (USD)")
plt.tight_layout()
plt.show()

print("Se observa la concentracion de viajes en Manhattan y areas circundantes.")

## Conclusiones

- La mayoria de las tarifas estan entre **$5 y $20 USD**
- Existen anomalias: tarifas negativas, coordenadas fuera de rango, pasajeros = 0
- La **distancia** es el factor mas determinante del precio
- Las tarifas son ligeramente mas altas en **horas pico** y **fines de semana**
- El numero de **pasajeros no afecta** significativamente la tarifa
- Los viajes se concentran en **Manhattan y areas circundantes de NYC**

### Siguientes pasos

Continua con el notebook `02_entrenamiento_uber.ipynb` para entrenar modelos con estas features.